# Importing Libraries

In [100]:
# Import polars for analytics
import polars as pl

# Path for data folder
from pathlib import Path

# Define Data Path

In [101]:
# Define data folder
data_folder = Path("../data")

# Loading the Dataset

In [102]:
# Load cleaned dataset
sales_df = pl.read_parquet(data_folder /"sales_cleaned.parquet")

# Preview dataset
sales_df.head()

transaction_id,customer_id,product_id,purchase_date,festival,city,product_name,category,customer_name,age,gender,price,revenue
str,str,str,datetime[μs],str,str,str,str,str,i64,str,i64,i64
"""T82""","""c377""","""p3""",2024-01-01 00:00:00,null,"""Jorhat""","""Lip Balm""","""Makeup""","""Harshil Varty""",21,"""Female""",149,149
"""T910""","""c202""","""p1""",2024-01-01 00:00:00,null,"""Indore""","""Vitamin C Serum""","""Skincare""","""Tanvi Chana""",51,"""Male""",499,499
"""T1273""","""c752""","""p2""",2024-01-01 00:00:00,null,"""Secunderabad""","""Herbal Shampoo""","""Haircare""","""Idika Roy""",48,"""Male""",199,199
"""T1369""","""c739""","""p0""",2024-01-01 00:00:00,null,"""Bhatpara""","""Aloe Vera Gel""","""Skincare""","""Harita Comar""",49,"""Female""",299,299
"""T2858""","""c169""","""p1""",2024-01-01 00:00:00,null,"""Bhagalpur""","""Vitamin C Serum""","""Skincare""","""Zayyan Mander""",51,"""Female""",499,499


# Product Performance

In [103]:
# Calculate product performance
product_performance = (
    sales_df.group_by(["product_id","product_name","category"])
    .agg(
        # number of transactions
        pl.len().alias("transactions"),

        # total revenue
        pl.col("revenue").sum().alias("total_revenue"),

        # average product price
        pl.col("revenue").mean().alias("avg_price")

    )
)

product_performance.head()

product_id,product_name,category,transactions,total_revenue,avg_price
str,str,str,u32,i64,f64
"""p8""","""Nail Polish""","""Makeup""",5234,1041566,199.0
"""p0""","""Aloe Vera Gel""","""Skincare""",5028,1503372,299.0
"""p4""","""Face Wash""","""Skincare""",5015,2000985,399.0
"""p9""","""Face Pack""","""Skincare""",5285,1580215,299.0
"""p5""","""Sunscreen SPF50""","""Skincare""",5056,3028544,599.0


# Sort Top and Bottom Products

In [104]:
# Sort top products by revenue
top_products = (product_performance.sort("total_revenue",descending=True))

top_products.head()

product_id,product_name,category,transactions,total_revenue,avg_price
str,str,str,u32,i64,f64
"""p5""","""Sunscreen SPF50""","""Skincare""",5056,3028544,599.0
"""p1""","""Vitamin C Serum""","""Skincare""",4943,2466557,499.0
"""p4""","""Face Wash""","""Skincare""",5015,2000985,399.0
"""p7""","""Body Lotion""","""Skincare""",4979,1737671,349.0
"""p9""","""Face Pack""","""Skincare""",5285,1580215,299.0


In [105]:
# Sort Bottom products by revenue
bottom_products = (product_performance.sort("total_revenue",descending=False))

bottom_products.head()

product_id,product_name,category,transactions,total_revenue,avg_price
str,str,str,u32,i64,f64
"""p3""","""Lip Balm""","""Makeup""",4671,695979,149.0
"""p2""","""Herbal Shampoo""","""Haircare""",4928,980672,199.0
"""p8""","""Nail Polish""","""Makeup""",5234,1041566,199.0
"""p6""","""Hair Oil""","""Haircare""",4861,1210389,249.0
"""p0""","""Aloe Vera Gel""","""Skincare""",5028,1503372,299.0


# Category Performance

In [106]:
# Revenue by category
category_performance = (
    sales_df.group_by(["category"])
    .agg(
        pl.len().alias("transactions"),
        
        pl.col("revenue").sum().alias("total_revenue"),

        pl.col("revenue").mean().alias("avg_order_value")
    )
    .sort("total_revenue", descending=True)
)

category_performance.sort("total_revenue",descending = True).head()

category,transactions,total_revenue,avg_order_value
str,u32,i64,f64
"""Skincare""",30306,12317344,406.432522
"""Haircare""",9789,2191061,223.82889
"""Makeup""",9905,1737545,175.420999


# City Performance

In [107]:
# Revenue by city
city_performance = (
    sales_df.group_by(["city"])
    .agg(
        pl.len().alias("transactions"),
        
        pl.col("revenue").sum().alias("total_revenue"),

        pl.col("revenue").mean().alias("avg_order_value")
    )
    .sort("total_revenue", descending=True)
)

city_performance.sort("avg_order_value",descending = True).head()

city,transactions,total_revenue,avg_order_value
str,u32,i64,f64
"""Bongaigaon""",38,14012,368.736842
"""Bhavnagar""",49,17751,362.265306
"""Motihari""",77,27873,361.987013
"""New Delhi""",83,29767,358.638554
"""Srikakulam""",78,27722,355.410256


In [108]:
city_performance.sort("total_revenue",descending = True).head()

city,transactions,total_revenue,avg_order_value
str,u32,i64,f64
"""South Dumdum""",446,144004,322.878924
"""Berhampore""",421,138479,328.928741
"""Allahabad""",435,137415,315.896552
"""Ghaziabad""",406,133544,328.926108
"""Haldia""",411,133489,324.790754


In [109]:
city_performance.sort("transactions",descending = True).head()

city,transactions,total_revenue,avg_order_value
str,u32,i64,f64
"""South Dumdum""",446,144004,322.878924
"""Allahabad""",435,137415,315.896552
"""Berhampore""",421,138479,328.928741
"""Haldia""",411,133489,324.790754
"""Ghaziabad""",406,133544,328.926108


## Pareto Analysis (80/20 Rule)
Checking wether few products drive most revenue

In [110]:
# Add revenue share 
products_pareto = (
    top_products.with_columns(
        (pl.col("total_revenue") / 
        pl.col("total_revenue").sum()
        ).alias("revenue_share")
    )
)

products_pareto.sort("revenue_share",descending = True)

product_id,product_name,category,transactions,total_revenue,avg_price,revenue_share
str,str,str,u32,i64,f64,f64
"""p5""","""Sunscreen SPF50""","""Skincare""",5056,3028544,599.0,0.186418
"""p1""","""Vitamin C Serum""","""Skincare""",4943,2466557,499.0,0.151826
"""p4""","""Face Wash""","""Skincare""",5015,2000985,399.0,0.123168
"""p7""","""Body Lotion""","""Skincare""",4979,1737671,349.0,0.10696
"""p9""","""Face Pack""","""Skincare""",5285,1580215,299.0,0.097268
"""p0""","""Aloe Vera Gel""","""Skincare""",5028,1503372,299.0,0.092538
"""p6""","""Hair Oil""","""Haircare""",4861,1210389,249.0,0.074504
"""p8""","""Nail Polish""","""Makeup""",5234,1041566,199.0,0.064112
"""p2""","""Herbal Shampoo""","""Haircare""",4928,980672,199.0,0.060364


## Revenue Share

In [111]:
product_performance = product_performance.with_columns(
    (pl.col("total_revenue") / pl.col("total_revenue").sum()*100).round(2).alias("revenue_share_pct" )
)

product_performance.sort("revenue_share_pct",descending = True)

product_id,product_name,category,transactions,total_revenue,avg_price,revenue_share_pct
str,str,str,u32,i64,f64,f64
"""p5""","""Sunscreen SPF50""","""Skincare""",5056,3028544,599.0,18.64
"""p1""","""Vitamin C Serum""","""Skincare""",4943,2466557,499.0,15.18
"""p4""","""Face Wash""","""Skincare""",5015,2000985,399.0,12.32
"""p7""","""Body Lotion""","""Skincare""",4979,1737671,349.0,10.7
"""p9""","""Face Pack""","""Skincare""",5285,1580215,299.0,9.73
"""p0""","""Aloe Vera Gel""","""Skincare""",5028,1503372,299.0,9.25
"""p6""","""Hair Oil""","""Haircare""",4861,1210389,249.0,7.45
"""p8""","""Nail Polish""","""Makeup""",5234,1041566,199.0,6.41
"""p2""","""Herbal Shampoo""","""Haircare""",4928,980672,199.0,6.04


# Adding cumulative column to the product table

In [112]:
# Sort products by revenue
product_performance = product_performance.sort(
    "total_revenue",descending=True    
)

# Add cumulative revenue column
product_performance = product_performance.with_columns(

    # cumulative revenue
    pl.col("total_revenue").cum_sum().alias("cumulative_revenue"),

    # cumulative revenue percentage
    (
        pl.col("total_revenue").cum_sum()
        /
        pl.col("total_revenue").sum()
        * 100
    ).round(2).alias("cumulative_revenue_pct")
)

# view table
product_performance


product_id,product_name,category,transactions,total_revenue,avg_price,revenue_share_pct,cumulative_revenue,cumulative_revenue_pct
str,str,str,u32,i64,f64,f64,i64,f64
"""p5""","""Sunscreen SPF50""","""Skincare""",5056,3028544,599.0,18.64,3028544,18.64
"""p1""","""Vitamin C Serum""","""Skincare""",4943,2466557,499.0,15.18,5495101,33.82
"""p4""","""Face Wash""","""Skincare""",5015,2000985,399.0,12.32,7496086,46.14
"""p7""","""Body Lotion""","""Skincare""",4979,1737671,349.0,10.7,9233757,56.84
"""p9""","""Face Pack""","""Skincare""",5285,1580215,299.0,9.73,10813972,66.56
"""p0""","""Aloe Vera Gel""","""Skincare""",5028,1503372,299.0,9.25,12317344,75.82
"""p6""","""Hair Oil""","""Haircare""",4861,1210389,249.0,7.45,13527733,83.27
"""p8""","""Nail Polish""","""Makeup""",5234,1041566,199.0,6.41,14569299,89.68
"""p2""","""Herbal Shampoo""","""Haircare""",4928,980672,199.0,6.04,15549971,95.72


## Save the Datasets

In [113]:
# Save product dataset
product_performance.write_parquet(data_folder/"product_performance.parquet")

# Save category dataset
category_performance.write_parquet(data_folder/"category_performance.parquet")

# Save city dataset
city_performance.write_parquet(data_folder/"city_performance.parquet")

# Save top products dataset
top_products.write_parquet(data_folder/"top_products.parquet")

In [114]:
top_products.head(10)

product_id,product_name,category,transactions,total_revenue,avg_price
str,str,str,u32,i64,f64
"""p5""","""Sunscreen SPF50""","""Skincare""",5056,3028544,599.0
"""p1""","""Vitamin C Serum""","""Skincare""",4943,2466557,499.0
"""p4""","""Face Wash""","""Skincare""",5015,2000985,399.0
"""p7""","""Body Lotion""","""Skincare""",4979,1737671,349.0
"""p9""","""Face Pack""","""Skincare""",5285,1580215,299.0
"""p0""","""Aloe Vera Gel""","""Skincare""",5028,1503372,299.0
"""p6""","""Hair Oil""","""Haircare""",4861,1210389,249.0
"""p8""","""Nail Polish""","""Makeup""",5234,1041566,199.0
"""p2""","""Herbal Shampoo""","""Haircare""",4928,980672,199.0


In [115]:
category_performance

category,transactions,total_revenue,avg_order_value
str,u32,i64,f64
"""Skincare""",30306,12317344,406.432522
"""Haircare""",9789,2191061,223.82889
"""Makeup""",9905,1737545,175.420999


In [116]:
product_performance.select(pl.col("total_revenue").sum())

total_revenue
i64
16245950
